In [7]:
import time
import gc
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

print("Loading dataset...")
dataset_train = load_dataset("clapAI/MultiLingualSentiment", split="train")
dataset_test = load_dataset("clapAI/MultiLingualSentiment", split="validation")

print("Đang lọc dữ liệu tiếng Anh...")
en_train = dataset_train.filter(lambda x: x["language"] == "en")
en_test = dataset_test.filter(lambda x: x["language"] == "en")

label_mapping = {"negative": 0, "neutral": 1, "positive": 2}

X_train_raw = en_train["text"]
y_train = np.array([label_mapping[label] for label in en_train["label"]], dtype=np.int32)

X_test_raw = en_test["text"]
y_test = np.array([label_mapping[label] for label in en_test["label"]], dtype=np.int32)

print(f"-> Tổng số mẫu Train tiếng Anh: {len(X_train_raw):,}")
print(f"-> Tổng số mẫu Test tiếng Anh: {len(X_test_raw):,}")

Loading dataset...
Đang lọc dữ liệu tiếng Anh...
-> Tổng số mẫu Train tiếng Anh: 1,215,709
-> Tổng số mẫu Test tiếng Anh: 152,225


In [8]:
standard_stop_words = text.ENGLISH_STOP_WORDS
negation_words = {'no', 'not', 'never', 'neither', 'nor', 'none', 'cannot', 'barely', 'hardly'}
safe_stop_words = list(standard_stop_words - negation_words)

sentiment_token_pattern = r'(?u)\b\w+\b|\?|!'

In [9]:
print("\n---Tiền xử lý văn bản và khởi tạo ma trận BoW ---")
start_time = time.time()

bow_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words=safe_stop_words,
    ngram_range=(1, 2),
    max_features=80000,
    token_pattern=sentiment_token_pattern,
    dtype=np.int32
)

X_train_bow = bow_vectorizer.fit_transform(X_train_raw)
X_test_bow = bow_vectorizer.transform(X_test_raw)

bow_time = time.time() - start_time
print(f"hoàn tất trong: {bow_time:.2f} giây.")


---Tiền xử lý văn bản và khởi tạo ma trận BoW ---
hoàn tất trong: 424.65 giây.


In [10]:
print("Naive Bayes + BoW...")
t0 = time.time()
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
nb_bow_time = time.time() - t0

nb_bow_preds = nb_bow.predict(X_test_bow)
nb_bow_acc = accuracy_score(y_test, nb_bow_preds)
nb_bow_f1 = f1_score(y_test, nb_bow_preds, average='macro')

print("Logistic Regression + BoW...")
t0 = time.time()
lr_bow = LogisticRegression(max_iter=500, tol=1e-3, n_jobs=-1, solver='lbfgs', random_state=42)
lr_bow.fit(X_train_bow, y_train)
lr_bow_time = time.time() - t0

lr_bow_preds = lr_bow.predict(X_test_bow)
lr_bow_acc = accuracy_score(y_test, lr_bow_preds)
lr_bow_f1 = f1_score(y_test, lr_bow_preds, average='macro')

del X_train_bow, X_test_bow
gc.collect()

Naive Bayes + BoW...
Logistic Regression + BoW...


227

In [11]:
print("\n--- TF-IDF ---")
start_time = time.time()

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words=safe_stop_words,
    ngram_range=(1, 2),
    max_features=80000,
    token_pattern=sentiment_token_pattern,
    dtype=np.float32
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

tfidf_time = time.time() - start_time
print(f"hoàn tất trong: {tfidf_time:.2f} giây.")


--- TF-IDF ---
hoàn tất trong: 419.05 giây.


In [13]:
print("Naive Bayes + TF-IDF...")
t0 = time.time()
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
nb_tfidf_time = time.time() - t0

nb_tfidf_preds = nb_tfidf.predict(X_test_tfidf)
nb_tfidf_acc = accuracy_score(y_test, nb_tfidf_preds)
nb_tfidf_f1 = f1_score(y_test, nb_tfidf_preds, average='macro')

print("Logistic Regression + TF-IDF...")
t0 = time.time()
lr_tfidf = LogisticRegression(max_iter=500, tol=1e-3, n_jobs=-1, solver='lbfgs', random_state=42)
lr_tfidf.fit(X_train_tfidf, y_train)
lr_tfidf_time = time.time() - t0

lr_tfidf_preds = lr_tfidf.predict(X_test_tfidf)
lr_tfidf_acc = accuracy_score(y_test, lr_tfidf_preds)
lr_tfidf_f1 = f1_score(y_test, lr_tfidf_preds, average='macro')

Naive Bayes + TF-IDF...
Logistic Regression + TF-IDF...


In [15]:
print("\n" + "="*75)
print(f"{'Cấu hình Mô Hình':<32} | {'Accuracy':<8} | {'F1-Score':<8} | {'Tốc độ Train':<14}")
print("="*75)
print(f"{'Naive Bayes + BoW':<32} | {nb_bow_acc:.4f} | {nb_bow_f1:.4f} | {nb_bow_time:.2f}s")
print(f"{'Logistic Regression + BoW':<32} | {lr_bow_acc:.4f} | {lr_bow_f1:.4f} | {lr_bow_time:.2f}s")
print(f"{'Naive Bayes + TF-IDF':<32} | {nb_tfidf_acc:.4f} | {nb_tfidf_f1:.4f} | {nb_tfidf_time:.2f}s")
print(f"{'Logistic Regression + TF-IDF':<32} | {lr_tfidf_acc:.4f} | {lr_tfidf_f1:.4f} | {lr_tfidf_time:.2f}s")
print("="*75)


Cấu hình Mô Hình                 | Accuracy | F1-Score | Tốc độ Train  
Naive Bayes + BoW                | 0.6874 | 0.6873 | 0.62s
Logistic Regression + BoW        | 0.7501 | 0.7454 | 217.46s
Naive Bayes + TF-IDF             | 0.7241 | 0.7203 | 0.52s
Logistic Regression + TF-IDF     | 0.7521 | 0.7480 | 41.55s
